In [1]:
# Install pysam
!pip install pysam


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 871.0 kB/s  0:00:09m0:00:0100:01


In [25]:
import pysam
import difflib
from collections import Counter
import math

# ============================================================
# CONFIGURATION
# ============================================================

REF_FASTA = "/Users/leandro/Desktop/github/NGS-data/ONT/RHv2HDR_241125/alleles.fasta"
BAM       = "/Users/leandro/Desktop/github/NGS-data/ONT/RHv2HDR_241125/HDR-only/aln.filtered.bam"

ref_name = "jPCR-RHv2"
cut_pos  = 749            # 0-based reference coordinate
window   = 50             # +/- bp around cut

FUZZY_THRESHOLD = 0.90    # raise to 0.95 for stricter HDR calling

expected_wt_seq  = "gctatCTATATATAGCTATCTATGTCTCCTTGGGGCCCAGACTGAGCACGTGATGGCAGAGGAAAGGAAGCCCTGCTTCCTCCAGAGGGacgttactggc"
expected_hdr_seq = "gctatCTATATATAGCTATCTATGTCTCCTTGGGGCCCAGACTGAGCACGGCAAGAGTTCCAGCCGGGCTATttacttttgtaaaactttatggtttgtg"

expected_wt  = expected_wt_seq.upper()
expected_hdr = expected_hdr_seq.upper()
expected_len = len(expected_hdr)

# ============================================================
# LOAD DATA
# ============================================================

bam = pysam.AlignmentFile(BAM, "rb")
ref = pysam.FastaFile(REF_FASTA)

win_start = cut_pos - window
win_end   = cut_pos + window + 1

# ============================================================
# HELPERS
# ============================================================

def read_has_indel_near_cut(read, cut_pos, window):
    """
    Detect insertion or deletion overlapping the cut window.
    """
    ref_pos = read.reference_start
    for op, length in read.cigartuples or []:
        if op in (0, 7, 8):           # M, =, X
            ref_pos += length
        elif op == 2:                # deletion
            if ref_pos < cut_pos + window and ref_pos + length > cut_pos - window:
                return True
            ref_pos += length
        elif op == 1:                # insertion
            if cut_pos - window <= ref_pos <= cut_pos + window:
                return True
        # soft/hard clip ignored
    return False


def get_read_seq_in_window(read, win_start, win_end):
    """
    Reconstruct read sequence aligned to reference window,
    including insertions and marking deletions as '-'.
    """
    seq = []
    read_seq = read.query_sequence or ""

    for qpos, rpos in read.get_aligned_pairs(matches_only=False):
        if rpos is not None and win_start <= rpos < win_end:
            if qpos is None:
                seq.append("-")              # deletion
            else:
                seq.append(read_seq[qpos])
        elif rpos is None and qpos is not None:
            # insertion: include for HDR sensitivity
            seq.append(read_seq[qpos])

    return "".join(seq).upper()

# ============================================================
# CLASSIFICATION
# ============================================================

counts = Counter()
debug  = Counter()
total  = 0

for read in bam.fetch(ref_name):

    # ---------------- basic filters ----------------
    if read.is_unmapped or read.is_secondary or read.is_supplementary:
        continue
    if read.mapping_quality < 20:
        continue
    if read.reference_start is None or read.reference_end is None:
        continue
    if not (read.reference_start <= win_end and read.reference_end >= win_start):
        continue

    total += 1

    # ---------------- NHEJ ----------------
    if read_has_indel_near_cut(read, cut_pos, window):
        counts["NHEJ"] += 1
        continue

    # ---------------- sequence reconstruction ----------------
    rseq = get_read_seq_in_window(read, win_start, win_end)
    rseq_clean = rseq.replace("-", "")

    if len(rseq_clean) < max(int(0.6 * expected_len), expected_len - 5):
        counts["Other"] += 1
        debug["too_short"] += 1
        continue

    # ---------------- HDR (exact) ----------------
    if expected_hdr in rseq_clean:
        counts["HDR"] += 1
        debug["exact_hdr"] += 1
        continue

    # ---------------- HDR (fuzzy) ----------------
    ratio = difflib.SequenceMatcher(None, expected_hdr, rseq_clean).ratio()
    if ratio >= FUZZY_THRESHOLD:
        counts["HDR"] += 1
        debug["fuzzy_hdr"] += 1
        continue

    # ---------------- WT ----------------
    if expected_wt in rseq_clean:
        counts["WT"] += 1
        continue

    counts["Other"] += 1

# ============================================================
# SUMMARY
# ============================================================

print("\n=== RESULTS ===")
for k in ["HDR", "WT", "NHEJ", "Other"]:
    print(f"{k}: {counts[k]}")
print("Total reads covering window:", total)

if total > 0:
    hdr_p = counts["HDR"] / total
    print(f"HDR % = {hdr_p * 100:.2f}%")
else:
    print("HDR % = 0.00%")

# Wilson 95% CI
if total > 0:
    z = 1.96
    p = counts["HDR"] / total
    center = (p + z*z/(2*total)) / (1 + z*z/total)
    half = (z * math.sqrt(p*(1-p)/total + z*z/(4*total*total))) / (1 + z*z/total)
    lower = max(0, center - half)
    upper = min(1, center + half)
    print(f"95% CI: {100*lower:.2f}% – {100*upper:.2f}%")

print("\nDebug breakdown:", dict(debug))

Reference window: agctatCTATATATAGCTATCTATGTCTCCTTGGGGCCCAGACTGAGCACGGCAAGAGTTCCAGCCGGGCTATttacttttgtaaaactttatggtttgtg
Expected HDR    : gctatCTATATATAGCTATCTATGTCTCCTTGGGGCCCAGACTGAGCACGGCAAGAGTTCCAGCCGGGCTATttacttttgtaaaactttatggtttgtg
Expected WT     : gctatCTATATATAGCTATCTATGTCTCCTTGGGGCCCAGACTGAGCACGTGATGGCAGAGGAAAGGAAGCCCTGCTTCCTCCAGAGGGacgttactggc

=== RESULTS ===
Other: 1094
NHEJ: 361
Total reads covering window: 1455
HDR % = 0.00%
95% CI: 0.00% – 0.26%
